From lecture 3:

The language of the model = probability of the sequence

Use n-grams (Markov assumption)

Looking at last n-1 items of the full hostory is enough

so for trigram:





$$ p(w_i \mid w_{i-2}, w_{i-1}) $$

Requirements

More than 2 languages and one in English

200k tokens or more for each

I will use Project Gutenberg as a source for the texts in following language

- German
- Spanish
- English

I will download books as plain text files and use them for the code



In [ ]:
!pip install sacremoses

In [ ]:
from sacremoses import MosesTokenizer

In [ ]:
tokenizer = MosesTokenizer()

def tokenize(text):
    return tokenizer.tokenize(text)

def get_stats(text):
    tokens = tokenize(text)
    return len(tokens), len(text.encode('utf-8'))

Upload the files
- english.txt
- german.txt
- czech.txt

to Collab and read them

In [ ]:
!wget https://raw.githubusercontent.com/isyaralin/NLP/main/english.txt
!wget https://raw.githubusercontent.com/isyaralin/NLP/main/spanish.txt
!wget https://raw.githubusercontent.com/isyaralin/NLP/main/german.txt

with open("english.txt", "r", encoding="utf-8") as f:
    english_text = f.read()

with open("spanish.txt", "r", encoding="utf-8") as f:
    spanish_text = f.read()

with open("german.txt", "r", encoding="utf-8") as f:
    german_text = f.read()

--2026-03-30 17:38:11--  https://raw.githubusercontent.com/isyaralin/NLP/main/english.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1227621 (1.2M) [text/plain]
Saving to: ‘english.txt.3’

english.txt.3       100%[===================>]   1.17M  --.-KB/s    in 0.008s  

2026-03-30 17:38:11 (138 MB/s) - ‘english.txt.3’ saved [1227621/1227621]

--2026-03-30 17:38:11--  https://raw.githubusercontent.com/isyaralin/NLP/main/spanish.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2205982 (2.1M) [text/plain]
Saving to: ‘spanish.t

In [ ]:
tokens_english = tokenize(english_text)
tokens_spanish = tokenize(spanish_text)
tokens_german = tokenize(german_text)

Now we read and tokenized the .txt files. Now we need to check the size

In [ ]:
print("English:", get_stats(english_text), "Bytes:", len(english_text.encode("utf-8")))
print("German:", get_stats(german_text), "Bytes:", len(german_text.encode("utf-8")))
print("Spanish:", get_stats(spanish_text), "Bytes:", len(spanish_text.encode("utf-8")))

English: (253930, 1206508) Bytes: 1206508
German: (211747, 1172195) Bytes: 1172195
Spanish: (450390, 2168309) Bytes: 2168309


In [ ]:
def calculate_oov_percentage(tokens, vocab):
    oov_count = 0
    for token in tokens:
        if token not in vocab:
            oov_count += 1
    return (oov_count / len(tokens)) * 100 if len(tokens) > 0 else 0


In [ ]:
print("English OOV Percentage:")
print(f"Heldout: {calculate_oov_percentage(heldout_english, vocab_english):.2f}%")
print(f"Test: {calculate_oov_percentage(test_english, vocab_english):.2f}%")

print("\nGerman OOV Percentage:")
print(f"Heldout: {calculate_oov_percentage(heldout_german, vocab_german):.2f}%")
print(f"Test: {calculate_oov_percentage(test_german, vocab_german):.2f}%")

print("\nSpanish OOV Percentage:")
print(f"Heldout: {calculate_oov_percentage(heldout_spanish, vocab_spanish):.2f}%")
print(f"Test: {calculate_oov_percentage(test_spanish, vocab_spanish):.2f}%")


English OOV Percentage:
Heldout: 5.66%
Test: 4.17%

German OOV Percentage:
Heldout: 18.43%
Test: 13.49%

Spanish OOV Percentage:
Heldout: 3.72%
Test: 3.95%


Since we have more than 200k tokens we can split the data now

Data =  (Train / Heldout / Test)


In [ ]:
def split_data(tokens):
    n = len(tokens)
    train = tokens[:int(0.8*n)]
    heldout = tokens[int(0.8*n):int(0.9*n)]
    test = tokens[int(0.9*n):]
    return train, heldout, test

train_english, heldout_english, test_english = split_data(tokens_english)
train_german, heldout_german, test_german = split_data(tokens_german)
train_spanish, heldout_spanish, test_spanish = split_data(tokens_spanish)

In [ ]:
def tokens_to_text(tokens):
    return " ".join(tokens)

train_text_english = tokens_to_text(train_english)
heldout_text_english = tokens_to_text(heldout_english)
test_text_english = tokens_to_text(test_english)

train_text_german = tokens_to_text(train_german)
heldout_text_german = tokens_to_text(heldout_german)
test_text_german = tokens_to_text(test_german)

train_text_spanish = tokens_to_text(train_spanish)
heldout_text_spanish = tokens_to_text(heldout_spanish)
test_text_spanish = tokens_to_text(test_spanish)

Now we count n-grams according to the lecture notes

In [ ]:
from collections import defaultdict

def count_ngrams(text, n):
    counts = defaultdict(int)
    for i in range(len(text) - n + 1):
        ngram = text[i:i+n]
        counts[ngram] += 1
    return counts

Building the Models
(Bigram and Trigram)

In [ ]:
# English
bigram_english = count_ngrams(train_text_english, 2)
trigram_english = count_ngrams(train_text_english, 3)

# German
bigram_german = count_ngrams(train_text_german, 2)
trigram_german = count_ngrams(train_text_german, 3)

# Spanish
bigram_spansih = count_ngrams(train_text_spanish, 2)
trigram_spanish = count_ngrams(train_text_spanish, 3)

Now we look for the top 5 trigrams

In [ ]:
def top_trigrams(trigram_counts):
    total = sum(trigram_counts.values())
    items = sorted(trigram_counts.items(), key=lambda x: x[1], reverse=True)

    for trigram, count in items[:5]:
        print(trigram, count, count/total)

In [ ]:
print("English:")
top_trigrams(trigram_english)

print("\nGerman:")
top_trigrams(trigram_german)

print("\nSpanish:")
top_trigrams(trigram_spanish)

English:
 th 19439 0.019547174061254984
 ,  15070 0.015153861469371501
the 14887 0.014969843111780594
he  12788 0.012859162605860834
ed  6649 0.006686000325802994

German:
en  19929 0.020931822693419012
 ,  15987 0.016791462160654812
er  12669 0.013306501164279465
ich 8613 0.009046404177751917
ch  8085 0.00849183533926904

Spanish:
 ,  32028 0.01841290720189119
 de 22466 0.012915710415813898
que 20934 0.012034963137391975
 qu 20228 0.011629083516918166
ue  19449 0.01118123617364749


From lecture 3:
Raw n-gram models have zero probabilities

So wee need to apply smoothing to not face this problem



In [ ]:
def smoothed_prob(trigram, trigram_counts, bigram_counts, vocab_size, delta):
    history = trigram[:2]
    return (trigram_counts[trigram] + delta) / (bigram_counts[history] + delta * vocab_size)

In [ ]:
vocab_english = set(train_english)
vocab_german = set(train_german)
vocab_spanish = set(train_spanish)

Vocab_english = len(vocab_english)
Vocab_german = len(vocab_german)
Vocab_spanish = len(vocab_spanish)

We need vocabulary size for the denominator and distributing probability mass from the smoothing formula thats why wee need this

In [ ]:
import math

def cross_entropy(text, trigram_counts, bigram_counts, vocab_size, delta):
    total = 0
    count = 0

    for i in range(len(text) - 2):
        trigram = text[i:i+3]
        p = smoothed_prob(trigram, trigram_counts, bigram_counts, vocab_size, delta)

        if p > 0:
            total += -math.log2(p)
            count += 1

    return total / count

In [ ]:
deltas = [0.01, 0.05, 0.1, 0.5]

def find_best_delta(heldout_text, trigram_counts, bigram_counts, vocab_size):
    best_delta = None
    best_score = float('inf')

    for d in deltas:
        score = cross_entropy(heldout_text, trigram_counts, bigram_counts, vocab_size, d)
        if score < best_score:
            best_score = score
            best_delta = d

    return best_delta

In [ ]:
delta_english = find_best_delta(heldout_text_english, trigram_english, bigram_english, Vocab_english)
delta_german = find_best_delta(heldout_text_german, trigram_german, bigram_german, Vocab_german)
delta_spanish = find_best_delta(heldout_text_spanish, trigram_spanish, bigram_spansih, Vocab_spanish)

print(f"English Delta: {delta_english}")
print(f"German Delta: {delta_german}")
print(f"Spanish Delta: {delta_spanish}")

English Delta: 0.01
German Delta: 0.01
Spanish Delta: 0.01


In [ ]:
print("English model on English test:", cross_entropy(test_text_english, trigram_english, bigram_english, Vocab_english, delta_english))
print("German model on German test:", cross_entropy(test_text_german, trigram_german, bigram_german, Vocab_german, delta_german))
print("Spanish model on Spanish test:", cross_entropy(test_text_spanish, trigram_spanish, bigram_spansih, Vocab_spanish, delta_spanish))

English model on English test: 2.874366167550094
German model on German test: 3.0485879731607772
Spanish model on Spanish test: 2.7359825259160804


In [ ]:
print("English model on German test:", cross_entropy(test_text_german, trigram_english, bigram_english, Vocab_english, delta_english))
print("English model on Spanish test:", cross_entropy(test_text_spanish, trigram_english, bigram_english, Vocab_english, delta_english))

print("German model on English test:", cross_entropy(test_text_english, trigram_german, bigram_german, Vocab_german, delta_german))
print("German model on Spanish test:", cross_entropy(test_text_spanish, trigram_german, bigram_german, Vocab_german, delta_german))

print("Spanish model on English test:", cross_entropy(test_text_english, trigram_spanish, bigram_spansih, Vocab_spanish, delta_spanish))
print("Spanish model on German test:", cross_entropy(test_text_german, trigram_spanish, bigram_spansih, Vocab_spanish, delta_spanish))

English model on German test: 5.5691635699485
English model on Spanish test: 5.25630416328236
German model on English test: 6.004427638165795
German model on Spanish test: 6.5196532112431855
Spanish model on English test: 8.040914158067334
Spanish model on German test: 7.794652811506109


In [ ]:
def identify_language(text):
    results = []

    ce_english = cross_entropy(text, trigram_english, bigram_english, Vocab_english, delta_english)
    ce_german = cross_entropy(text, trigram_german, bigram_german, Vocab_german, delta_german)
    ce_spanish = cross_entropy(text, trigram_spanish, bigram_spansih, Vocab_spanish, delta_spanish)

    results.append((2**(-ce_english), "English"))
    results.append((2**(-ce_german), "German"))
    results.append((2**(-ce_spanish), "Spanish"))

    results.sort(reverse=True)
    return results

In [ ]:
identify_language("This is a English text sentence to test")

[(0.09892443377148816, 'English'),
 (0.01865533865447941, 'Spanish'),
 (0.014928143538225847, 'German')]